In [5]:
import json
import numpy as np
import os
import cv2
import rasterio
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, ZeroPadding2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy
from sklearn.model_selection import train_test_split

print("Starting script...")

# Load and parse JSON file
json_file_path = r"C:\Users\SARTHAK\Desktop\train_annotations.json"
images_dir = r"C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images"

print(f"Loading JSON file from: {json_file_path}")
with open(json_file_path, 'r') as f:
    data = json.load(f)
    print("JSON file loaded successfully!")

def preprocess_image_and_mask(image_file, annotations, img_shape=(128, 128)):
    if not os.path.exists(image_file):
        print(f"Error: Image file {image_file} not found.")
        return None, None

    with rasterio.open(image_file) as img:
        red_band = img.read(1)
        nir_band = img.read(4)
        composite_image = np.stack([red_band, nir_band, np.zeros_like(red_band)], axis=-1)

    image = cv2.resize(composite_image, img_shape)
    image = image / 255.0

    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    for annotation in annotations:
        points = np.array(annotation['segmentation'], dtype=np.int32).reshape(-1, 2)
        cv2.fillPoly(mask, [points], 1)

    return image, mask

images = []
masks = []
failed_images = []

for img_data in data['images']:
    img_file = img_data['file_name']
    annotations = img_data['annotations']
    img_file_path = os.path.join(images_dir, img_file)
    print(f"Processing image: {img_file_path}")

    if not os.path.exists(img_file_path):
        print(f"Error: Image file {img_file_path} not found.")
        failed_images.append(img_file_path)
        continue

    image, mask = preprocess_image_and_mask(img_file_path, annotations)
    if image is not None and mask is not None:
        images.append(image)
        masks.append(mask)
    else:
        print(f"Skipping image {img_file_path} due to loading issues.")
        failed_images.append(img_file_path)

images = np.array(images)
masks = np.array(masks)

print(f"Parsed {len(images)} images and their corresponding masks.")
print(f"Failed to parse {len(failed_images)} images.")

if failed_images:
    print("Failed images:")
    for failed_image in failed_images:
        print(f" - {failed_image}")

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(images, masks, test_size=0.2, random_state=42)

# Define the U-Net model
def unet_model(input_size=(128, 128, 3)):
    inputs = Input(input_size)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    up4 = UpSampling2D(size=(2, 2))(pool3)
    up4 = ZeroPadding2D(((0, conv2.shape[1] - up4.shape[1]), (0, conv2.shape[2] - up4.shape[2])))(up4)
    up4 = concatenate([up4, conv2], axis=3)
    conv4 = Conv2D(128, 3, activation='relu', padding='same')(up4)
    conv4 = Conv2D(128, 3, activation='relu', padding='same')(conv4)

    up5 = UpSampling2D(size=(2, 2))(conv4)
    up5 = ZeroPadding2D(((0, conv1.shape[1] - up5.shape[1]), (0, conv1.shape[2] - up5.shape[2])))(up5)
    up5 = concatenate([up5, conv1], axis=3)
    conv5 = Conv2D(64, 3, activation='relu', padding='same')(up5)
    conv5 = Conv2D(64, 3, activation='relu', padding='same')(conv5)

    conv6 = Conv2D(1, 1, activation='sigmoid')(conv5)

    model = Model(inputs=[inputs], outputs=[conv6])

    return model

print("Building U-Net model...")
model = unet_model()
model.compile(optimizer=Adam(), loss=binary_crossentropy, metrics=['accuracy', tf.keras.metrics.MeanIoU(num_classes=2)])
print("Model compiled successfully!")

# Train the model
print("Training the model...")
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=16)

# Evaluate the model
print("Evaluating the model...")
score = model.evaluate(X_val, y_val)
print(f'Test loss: {score[0]}')
print(f'Test accuracy: {score[1]}')

# Save the model
print("Saving the model...")
model.save('segmentation_model.h5')
print("Model saved successfully!")

Starting script...
Loading JSON file from: C:\Users\SARTHAK\Desktop\train_annotations.json
JSON file loaded successfully!
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_0.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_1.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_10.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_100.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_101.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_102.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_103.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_104.tif
Processing image: C:\Users\SARTHAK\Desktop\SolafuneScripts\Train_images\train_images\train_

C:\Users\SARTHAK\anaconda3\Lib\site-packages\keras\src\models\functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_63']
Received: inputs=Tensor(shape=(None, 128, 128, 3))
  warnings.warn(msg)


9/9 ━━━━━━━━━━━━━━━━━━━━ 65s 6s/step - accuracy: 0.4692 - loss: 0.6947 - mean_io_u_3: 0.2758 - val_accuracy: 0.5271 - val_loss: 0.6927 - val_mean_io_u_3: 0.2636
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 80s 6s/step - accuracy: 0.5713 - loss: 0.6897 - mean_io_u_3: 0.2856 - val_accuracy: 0.5271 - val_loss: 0.6920 - val_mean_io_u_3: 0.2636
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 83s 6s/step - accuracy: 0.5853 - loss: 0.6794 - mean_io_u_3: 0.2927 - val_accuracy: 0.5271 - val_loss: 0.7035 - val_mean_io_u_3: 0.2636
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 89s 10s/step - accuracy: 0.5989 - loss: 0.6759 - mean_io_u_3: 0.2994 - val_accuracy: 0.5271 - val_loss: 0.6952 - val_mean_io_u_3: 0.2636
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 56s 6s/step - accuracy: 0.6239 - loss: 0.6687 - mean_io_u_3: 0.3119 - val_accuracy: 0.5271 - val_loss: 0.7019 - val_mean_io_u_3: 0.2636
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 55s 6s/step - accuracy: 0.5667 - loss: 0.6877 - mean_io_u_3: 0.2833 - val_accuracy: 0.5271 - val_loss: 0.7038 - va

Test loss: 0.7011510729789734
Test accuracy: 0.5271012783050537
Saving the model...
Model saved successfully!
